In [1]:
import pandas as pd
import optuna
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced (Secondly) Dataset.csv')

# Drop diseases with less than 1000 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 1000].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 37
Number of rows left: 44748


In [4]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune for KNN
    n_neighbors = trial.suggest_int('n_neighbors', 3, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    p = trial.suggest_int('p', 1, 2)  # p=1 (Manhattan), p=2 (Euclidean)

    # Create KNeighborsClassifier with hyperparameters
    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        p=p
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_neighbors={n_neighbors}, weights={weights}, p={p}, Accuracy={accuracy:.4f}")

    return accuracy

In [5]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize', 
    study_name="knn_diseases_symptoms_dropextremelymore1000withoutSMOTE_secondreductionstudy",
    storage=r"sqlite:///C:/Users/khiew/Downloads/knn.db", 
    load_if_exists=True
)

# Optimize the study with your objective function, adjust n_trials as needed
study.optimize(objective, n_trials=20)

# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

# After finding the best hyperparameters, you can fit the final KNN model on resampled data
best_params = study.best_trial.params
final_model = KNeighborsClassifier(
    n_neighbors=best_params['n_neighbors'],
    weights=best_params['weights'],
    p=best_params['p']
)
final_model.fit(X_train, y_train)

# Evaluate on test set
y_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")


[I 2025-04-27 16:43:29,124] A new study created in RDB with name: knn_diseases_symptoms_dropextremelymore1000withoutSMOTE_secondreductionstudy
[I 2025-04-27 16:44:03,599] Trial 0 finished with value: 0.4497736333762791 and parameters: {'n_neighbors': 42, 'weights': 'uniform', 'p': 1}. Best is trial 0 with value: 0.4497736333762791.


Trial 0: n_neighbors=42, weights=uniform, p=1, Accuracy=0.4498


[I 2025-04-27 16:44:28,985] Trial 1 finished with value: 0.45072341257361714 and parameters: {'n_neighbors': 23, 'weights': 'distance', 'p': 2}. Best is trial 1 with value: 0.45072341257361714.


Trial 1: n_neighbors=23, weights=distance, p=2, Accuracy=0.4507


[I 2025-04-27 16:44:54,037] Trial 2 finished with value: 0.4410863303682282 and parameters: {'n_neighbors': 13, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.45072341257361714.


...

Trial 17: n_neighbors=45, weights=distance, p=1, Accuracy=0.4567


[I 2025-04-27 16:51:58,044] Trial 18 finished with value: 0.4488520992835522 and parameters: {'n_neighbors': 47, 'weights': 'uniform', 'p': 2}. Best is trial 16 with value: 0.45955080178015556.


Trial 18: n_neighbors=47, weights=uniform, p=2, Accuracy=0.4489


[I 2025-04-27 16:52:18,929] Trial 19 finished with value: 0.4477348783927096 and parameters: {'n_neighbors': 16, 'weights': 'distance', 'p': 2}. Best is trial 16 with value: 0.45955080178015556.


Trial 19: n_neighbors=16, weights=distance, p=2, Accuracy=0.4477

Best Trial:
FrozenTrial(number=16, state=TrialState.COMPLETE, values=[0.45955080178015556], datetime_start=datetime.datetime(2025, 4, 27, 16, 50, 35, 834351), datetime_complete=datetime.datetime(2025, 4, 27, 16, 51, 1, 932879), params={'n_neighbors': 50, 'weights': 'distance', 'p': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_neighbors': IntDistribution(high=50, log=False, low=3, step=1), 'weights': CategoricalDistribution(choices=('uniform', 'distance')), 'p': IntDistribution(high=2, log=False, low=1, step=1)}, trial_id=246, value=None)
Best Hyperparameters:
{'n_neighbors': 50, 'weights': 'distance', 'p': 2}
Test Accuracy: 0.4644


In [ ]:
import matplotlib.pyplot as plt

# Load existing study from the database
study = optuna.load_study(
    study_name="knn_diseases_symptoms_dropextremelymore1000withoutSMOTE_secondreductionstudy",
    storage="sqlite:///C:/Users/khiew/Downloads/knn.db"
)

# 1. Create model with best parameters
best_params = study.best_trial.params
# ============================
# Simulate Training Lifecycle
# ============================

# We will vary `n_neighbors` to see how accuracy changes
n_neighbors_list = list(range(3, best_params['n_neighbors']+10, 2))  # Varying n_neighbors
train_accuracies = []
test_accuracies = []

for n in n_neighbors_list:
    model = KNeighborsClassifier(
        n_neighbors=n,
        weights=best_params['weights'],
        p=best_params['p']
    )
    
    model.fit(X_train, y_train)

    # Train accuracy
    y_train_pred = model.predict(X_train)
    train_acc = accuracy_score(y_train, y_train_pred)

    # Test accuracy
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

# Plotting the training lifecycle (Accuracy vs n_neighbors)
plt.figure(figsize=(10,6))
plt.plot(n_neighbors_list, train_accuracies, label='Train Accuracy', marker='o')
plt.plot(n_neighbors_list, test_accuracies, label='Test Accuracy', marker='s')
plt.xlabel('Number of Neighbors (n_neighbors)')
plt.ylabel('Accuracy')
plt.title('Training Lifecycle - Accuracy vs Number of Neighbors')
plt.legend()
plt.grid(True)
plt.show()